In [52]:
import gradio as gr
import pandas as pd
import numpy as np
from io import BytesIO
import matplotlib.pyplot as plt
import datetime
import time
import os
import json
import warnings
warnings.filterwarnings('ignore')

# SQLite Database
from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime, Boolean, Text
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

# ML Models
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    PROPHET_AVAILABLE = False
    print("⚠️ Prophet not installed. Using fallback forecasting.")

# ==================== DATABASE SETUP ====================

DATABASE_URL = "sqlite:///./msme_data.db"
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

class MSMEProfile(Base):
    __tablename__ = "msme_profiles"

    id = Column(Integer, primary_key=True, index=True)
    mobile_number = Column(String(15), unique=True, index=True)
    full_name = Column(String(100))
    email = Column(String(100))
    role = Column(String(50))
    company_name = Column(String(200))
    business_type = Column(String(50))
    state = Column(String(50))
    city = Column(String(100))
    years_operation = Column(Integer)
    monthly_revenue_range = Column(String(50))
    verification_status = Column(String(20), default="PENDING")
    created_at = Column(DateTime, default=datetime.datetime.utcnow)
    consent_given = Column(Boolean, default=False)

class AnalyticsResult(Base):
    __tablename__ = "analytics_results"

    id = Column(Integer, primary_key=True)
    msme_id = Column(Integer)
    analysis_date = Column(DateTime, default=datetime.datetime.utcnow)
    total_sales = Column(Float)
    total_profit = Column(Float)
    msme_health_score = Column(Float)
    forecast_6m = Column(Text)
    forecast_12m = Column(Text)

# Initialize database
Base.metadata.create_all(bind=engine)

def get_db():
    db = SessionLocal()
    try:
        return db
    finally:
        db.close()

# ==================== DATABASE OPERATIONS ====================

def save_user_profile(profile_data):
    """Save user profile to database"""
    db = SessionLocal()
    try:
        existing = db.query(MSMEProfile).filter(
            MSMEProfile.mobile_number == profile_data['mobile_number']
        ).first()

        if existing:
            for key, value in profile_data.items():
                setattr(existing, key, value)
            db.commit()
            return existing.id
        else:
            profile = MSMEProfile(**profile_data)
            db.add(profile)
            db.commit()
            db.refresh(profile)
            return profile.id
    finally:
        db.close()

def get_user_profile(mobile_number):
    """Get user profile from database"""
    db = SessionLocal()
    try:
        profile = db.query(MSMEProfile).filter(
            MSMEProfile.mobile_number == mobile_number
        ).first()

        if profile:
            return {
                'id': profile.id,
                'mobile_number': profile.mobile_number,
                'full_name': profile.full_name,
                'company_name': profile.company_name,
                'business_type': profile.business_type,
                'state': profile.state,
                'city': profile.city,
                'verification_status': profile.verification_status
            }
        return None
    finally:
        db.close()

# ==================== ML MODELS ====================

def normalize(series):
    """Normalize series to 0-1 range"""
    if series.empty or series.max() == series.min():
        return pd.Series(0, index=series.index)
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

def calculate_scores(df):
    """Calculate risk and performance scores"""
    # Ensure numeric columns
    numeric_cols = ['Monthly_Sales_INR', 'Monthly_Operating_Cost_INR', 'Outstanding_Loan_INR',
                   'Vendor_Delivery_Reliability', 'Inventory_Turnover', 'Avg_Margin_Percent',
                   'Monthly_Demand_Units']

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # Avoid division by zero
    df['Monthly_Sales_INR'] = df['Monthly_Sales_INR'].replace(0, 1)

    # Calculate scores
    df["Cashflow_Stress"] = normalize(df["Monthly_Operating_Cost_INR"] / df["Monthly_Sales_INR"])
    df["Loan_Stress"] = normalize(df["Outstanding_Loan_INR"] / (df["Monthly_Sales_INR"] * 12))
    df["Financial_Risk_Score"] = (0.5 * df["Cashflow_Stress"] + 0.5 * df["Loan_Stress"]).clip(0, 1)

    df["Vendor_Score"] = (
        0.5 * df["Vendor_Delivery_Reliability"] +
        0.3 * normalize(df["Inventory_Turnover"]) +
        0.2 * normalize(df["Avg_Margin_Percent"])
    ).clip(0, 1)

    df["Growth_Potential_Score"] = (
        0.4 * normalize(df["Monthly_Demand_Units"]) +
        0.35 * normalize(df["Avg_Margin_Percent"]) +
        0.25 * normalize(df.get("Digital_Ad_Spend_INR", pd.Series(0)))
    ).clip(0, 1)

    df["MSME_Health_Score"] = (
        (1 - df["Financial_Risk_Score"]) * 0.4 +
        df["Vendor_Score"] * 0.3 +
        df["Growth_Potential_Score"] * 0.3
    ) * 100

    return df

def forecast_sales(df):
    """Real sales forecasting using Prophet or fallback"""
    try:
        if PROPHET_AVAILABLE and 'Date' in df.columns:
            # Prepare data for Prophet
            ts_data = df.groupby('Date')['Monthly_Sales_INR'].sum().reset_index()
            ts_data.columns = ['ds', 'y']
            ts_data['ds'] = pd.to_datetime(ts_data['ds'])

            if len(ts_data) < 2:
                raise ValueError("Insufficient data for Prophet")

            # Train Prophet model
            model = Prophet(
                yearly_seasonality=True,
                weekly_seasonality=False,
                daily_seasonality=False,
                changepoint_prior_scale=0.05
            )
            model.fit(ts_data)

            # 6-month forecast
            future_6m = model.make_future_dataframe(periods=180)
            forecast_6m = model.predict(future_6m)

            # 12-month forecast
            future_12m = model.make_future_dataframe(periods=365)
            forecast_12m = model.predict(future_12m)

            return {
                '6_month': {
                    'forecast': forecast_6m['yhat'].tail(180).sum(),
                    'lower': forecast_6m['yhat_lower'].tail(180).sum(),
                    'upper': forecast_6m['yhat_upper'].tail(180).sum()
                },
                '12_month': {
                    'forecast': forecast_12m['yhat'].tail(365).sum(),
                    'lower': forecast_12m['yhat_lower'].tail(365).sum(),
                    'upper': forecast_12m['yhat_upper'].tail(365).sum()
                }
            }
        else:
            raise ValueError("Using fallback forecasting")

    except Exception as e:
        print(f"Prophet forecasting failed: {str(e)}. Using fallback.")
        # Fallback: Simple growth-based forecast
        total_sales = df['Monthly_Sales_INR'].sum()
        avg_growth = 0.05  # 5% growth assumption

        forecast_6m = total_sales * 6 * (1 + avg_growth)
        forecast_12m = total_sales * 12 * (1 + avg_growth)

        return {
            '6_month': {
                'forecast': forecast_6m,
                'lower': forecast_6m * 0.85,
                'upper': forecast_6m * 1.15
            },
            '12_month': {
                'forecast': forecast_12m,
                'lower': forecast_12m * 0.85,
                'upper': forecast_12m * 1.15
            }
        }

def segment_customers(df):
    """K-Means customer segmentation"""
    try:
        if 'SKU_Name' not in df.columns:
            return None

        # RFM calculation
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'])
            reference_date = df['Date'].max()

            rfm = df.groupby('SKU_Name').agg({
                'Date': lambda x: (reference_date - x.max()).days,
                'Monthly_Sales_INR': ['count', 'sum']
            })

            rfm.columns = ['recency', 'frequency', 'monetary']

            if len(rfm) >= 3:
                # K-Means clustering
                scaler = StandardScaler()
                rfm_scaled = scaler.fit_transform(rfm)

                n_clusters = min(5, len(rfm))
                kmeans = KMeans(n_clusters=n_clusters, random_state=42)
                rfm['segment'] = kmeans.fit_predict(rfm_scaled)

                segment_names = ['Champions', 'Loyal', 'Potential', 'At Risk', 'Lost']
                rfm['segment_name'] = rfm['segment'].apply(
                    lambda x: segment_names[x] if x < len(segment_names) else f'Segment {x}'
                )

                return rfm['segment_name'].value_counts().to_dict()

        return None
    except Exception as e:
        print(f"Segmentation error: {str(e)}")
        return None

# ==================== ANALYTICS FUNCTIONS ====================

def generate_insights(user_data, df):
    """Generate AI-powered insights"""
    try:
        df = calculate_scores(df)

        top_skus = df.nlargest(5, 'Monthly_Sales_INR')[
            ['SKU_Name', 'Monthly_Sales_INR', 'Monthly_Demand_Units']
        ]

        insights = f"""
## 🎯 DataNetra AI Insights for {user_data['company_name']}

### 📊 Overall Performance Metrics
- **Average Financial Risk Score:** {df["Financial_Risk_Score"].mean():.2f}
- **Average Vendor Score:** {df["Vendor_Score"].mean():.2f}
- **Average Growth Potential Score:** {df["Growth_Potential_Score"].mean():.2f}
- **Overall MSME Health Score:** {df["MSME_Health_Score"].mean():.2f}%

### 🏆 Top 5 Performing SKUs
{top_skus.to_markdown(index=False)}

### 💡 AI Recommendations
1. **Focus on High-Performers:** Prioritize inventory for top 5 SKUs
2. **Risk Management:** Monitor SKUs with Financial Risk Score > 0.7
3. **Vendor Optimization:** Strengthen relationships with high-reliability vendors
4. **Demand Planning:** Use ML forecasts to optimize stock levels
"""

        # Run ML forecasting
        forecast_results = forecast_sales(df)

        insights += f"""

### 📈 ML-Powered Sales Forecast
**6-Month Forecast:**
- Predicted Sales: ₹{forecast_results['6_month']['forecast']:,.0f}
- Range: ₹{forecast_results['6_month']['lower']:,.0f} - ₹{forecast_results['6_month']['upper']:,.0f}

**12-Month Forecast:**
- Predicted Sales: ₹{forecast_results['12_month']['forecast']:,.0f}
- Range: ₹{forecast_results['12_month']['lower']:,.0f} - ₹{forecast_results['12_month']['upper']:,.0f}
"""

        # Customer segmentation
        segments = segment_customers(df)
        if segments:
            insights += "\n### 👥 Customer Segments (K-Means Clustering)\n"
            for segment, count in segments.items():
                insights += f"- {segment}: {count} products\n"

        return insights, None, forecast_results

    except Exception as e:
        return None, f"Error generating insights: {str(e)}", None

def generate_dashboard_data(user_data, df):
    """Generate dashboard KPIs and charts"""
    try:
        df = calculate_scores(df)

        total_sales = df['Monthly_Sales_INR'].sum()
        avg_margin = df['Avg_Margin_Percent'].mean() if 'Avg_Margin_Percent' in df.columns else 0
        total_profit = total_sales * (avg_margin / 100)
        health_score = df['MSME_Health_Score'].mean()
        growth_score = df['Growth_Potential_Score'].mean()

        # Chart 1: Top SKUs
        fig1, ax1 = plt.subplots(figsize=(8, 5))
        top_skus = df.nlargest(10, 'Monthly_Sales_INR')
        ax1.barh(top_skus['SKU_Name'], top_skus['Monthly_Sales_INR'], color='steelblue')
        ax1.set_xlabel('Sales (INR)')
        ax1.set_title('Top 10 Products by Sales')
        plt.tight_layout()

        # Chart 2: Score Distribution
        fig2, ax2 = plt.subplots(figsize=(8, 5))
        scores = ['Financial Risk', 'Vendor Score', 'Growth Potential']
        values = [
            df['Financial_Risk_Score'].mean(),
            df['Vendor_Score'].mean(),
            df['Growth_Potential_Score'].mean()
        ]
        ax2.bar(scores, values, color=['red', 'green', 'blue'])
        ax2.set_ylabel('Score (0-1)')
        ax2.set_title('Performance Scores Overview')
        ax2.set_ylim(0, 1)
        plt.tight_layout()

        # Chart 3: Sales vs Margin
        fig3, ax3 = plt.subplots(figsize=(8, 5))
        ax3.scatter(df['Monthly_Sales_INR'], df['Avg_Margin_Percent'],
                   alpha=0.6, c=df['MSME_Health_Score'], cmap='RdYlGn')
        ax3.set_xlabel('Monthly Sales (INR)')
        ax3.set_ylabel('Margin (%)')
        plt.colorbar(ax3.collections[0], label='Health Score')
        plt.tight_layout()

        # Chart 4: Forecast visualization
        fig4, ax4 = plt.subplots(figsize=(8, 5))
        forecast_results = forecast_sales(df)
        months = ['6M', '12M']
        forecasts = [
            forecast_results['6_month']['forecast'],
            forecast_results['12_month']['forecast']
        ]
        ax4.bar(months, forecasts, color='purple', alpha=0.7)
        ax4.set_ylabel('Forecasted Sales (INR)')
        ax4.set_title('Sales Forecast (Prophet ML Model)')
        plt.tight_layout()

        # Removed plt.close('all') to keep figures open for PDF generation and Gradio display

        return (
            f"### 💰 Total Sales\n₹{total_sales:,.0f}",
            f"### 📈 Total Profit\n₹{total_profit:,.0f}",
            f"### 🧠 Health Score\n{health_score:.1f}%",
            f"### 🚀 Growth Score\n{growth_score:.2f}",
            fig1, fig2, fig3, fig4,
            None # Error message placeholder
        )

    except Exception as e:
        return (
            "N/A", "N/A", "N/A", "N/A",
            None, None, None, None,
            f"Error: {str(e)}"
        )

def generate_pdf_report(user_data, df, dashboard_figs=None, dashboard_error_message=None):
    """Generate PDF report"""
    try:
        from matplotlib.backends.backend_pdf import PdfPages

        df = calculate_scores(df)
        forecast_results = forecast_sales(df)

        pdf_path = f"/tmp/{user_data['company_name']}_report.pdf".replace(" ", "_")

        with PdfPages(pdf_path) as pdf:
            # Cover page
            plt.figure(figsize=(8, 6))
            plt.axis('off')
            plt.text(0.5, 0.6, "MSME Analytics Report",
                    fontsize=24, ha='center', weight='bold')
            plt.text(0.5, 0.5, f"{user_data['company_name']}",
                    fontsize=16, ha='center')
            plt.text(0.5, 0.4, f"{datetime.datetime.now().strftime('%d %B %Y')}",
                    fontsize=12, ha='center')
            pdf.savefig()
            plt.close()

            # Summary page
            plt.figure(figsize=(8, 6))
            plt.axis('off')
            summary_text = f"""
Executive Summary

Total Sales: ₹{df['Monthly_Sales_INR'].sum():,.0f}
Average Health Score: {df['MSME_Health_Score'].mean():.1f}%
Average Risk Score: ₹{df['Financial_Risk_Score'].mean():.2f}

6-Month Forecast: ₹{forecast_results['6_month']['forecast']:,.0f}
12-Month Forecast: ₹{forecast_results['12_month']['forecast']:,.0f}

Top Product: {df.nlargest(1, 'Monthly_Sales_INR')['SKU_Name'].values[0]}
"""
            if dashboard_error_message:
                summary_text += f"\n\n**Warning: Dashboard chart generation failed for PDF: {dashboard_error_message}**"

            plt.text(0.1, 0.5, summary_text, fontsize=12, family='monospace')
            pdf.savefig()
            plt.close()

            # Charts
            if dashboard_figs is not None:
                for fig in [dashboard_figs[0], dashboard_figs[1], dashboard_figs[2], dashboard_figs[3]]: # Only pass figures
                    if fig is not None:
                        pdf.savefig(fig)

        # Check if PDF file was actually created and is not empty
        if not os.path.exists(pdf_path) or os.path.getsize(pdf_path) == 0:
            return None, "PDF generation failed: The generated file is empty or missing. This might indicate an issue with the data or plotting backend."

        return pdf_path, None

    except Exception as e:
        return None, f"PDF Error: {str(e)}"

# ==================== GRADIO UI ====================

states_cities = {
    "Choose State": [], # Added default placeholder
    "Kerala": ["Kochi", "Thiruvananthapuram", "Kozhikode", "Other"],
    "Tamil Nadu": ["Chennai", "Madurai", "Coimbatore", "Other"],
    "Karnataka": ["Bangalore", "Mysore", "Mangalore", "Other"],
    "Telangana": ["Hyderabad", "Warangal", "Other"],
    "Andhra Pradesh": ["Visakhapatnam", "Vijayawada", "Other"],
    "Delhi": ["New Delhi", "Other"]
}

business_types = ["Choose Business Type", "FMCG", "Supermarket", "Clothing", "Electronics"]
roles = ["Business Owner", "Co-Founder", "Manager", "Analyst", "Store Manager"]

with gr.Blocks(title="MSME Intelligent Agent") as demo:

    step_state = gr.State(0)
    user_id_state = gr.State(0)
    user_data_state = gr.State({}) # Stores dict of user profile data

    # State variables to hold data across steps
    user_name_state = gr.State("")
    mobile_state = gr.State("")
    company_state = gr.State("")
    business_type_state = gr.State("")
    state_state = gr.State("")
    city_state = gr.State("")
    other_city_input_state = gr.State("") # For 'Other' city input

    analysis_summary_state = gr.State({}) # New state to hold dynamic data for storyboard

    # Landing Page (Step 0)
    with gr.Column(visible=True) as step0_landing_col:
        gr.Markdown("# 🚀 MSME IntelliCore")
        gr.Markdown("### AI-Powered Business Intelligence for MSMEs")

        gr.Markdown("## 🔐 Already Registered on This Platform")
        login_mobile = gr.Textbox(label="Mobile Number")
        login_btn = gr.Button("Login/Check Status", variant="primary")
        login_status = gr.Markdown()

        gr.Markdown("## ✨ First-Time MSME User on This Platform")
        register_btn = gr.Button("Register New Account", variant="secondary")

    # Step 1: User Information
    with gr.Column(visible=False) as step1_user_info_col:
        gr.Markdown("## Step 1: User Information")
        name_input = gr.Textbox(label="Full Name*")
        mobile_input = gr.Textbox(label="Mobile Number*")
        email_input = gr.Textbox(label="Email")
        role_input = gr.Dropdown(choices=roles, label="Role*")

        with gr.Row():
            cancel1_btn = gr.Button("Cancel")
            next1_btn = gr.Button("Next →", variant="primary")
        error1 = gr.Markdown()

    # Step 2: MSME Verification (formerly Step 3)
    with gr.Column(visible=False) as step2_verification_col:
        gr.Markdown("## Step 2: MSME Verification")
        msme_checkbox = gr.Checkbox(label="I am a registered MSME", value=False)
        certificate_file = gr.File(label="Upload MSME Certificate (Mandatory)", file_types=['.pdf', '.doc', '.docx'])
        consent_checkbox = gr.Checkbox(label="I consent to share my MSME details for verification", value=False)

        with gr.Row():
            back2_btn = gr.Button("← Back")
            submit_btn_step2 = gr.Button("Submit for Verification", variant="primary")
        status_output_step2 = gr.Markdown() # Renamed output
        next_step_btn_step2 = gr.Button("Continue to Business Profile →", visible=False, variant="primary") # Renamed button

    # Step 3: Business Profile (formerly Step 2)
    with gr.Column(visible=False) as step3_business_profile_col:
        gr.Markdown("## Step 3: Business Profile")
        company_input = gr.Textbox(label="Company Name*")
        business_type_input = gr.Dropdown(choices=business_types, label="Business Type*", value="Choose Business Type")
        state_input = gr.Dropdown(choices=list(states_cities.keys()), label="State*", value="Choose State")
        city_input = gr.Dropdown(choices=["Choose City"], label="City*", value="Choose City") # Default choices including placeholder
        years_input = gr.Number(label="Years in Operation", value=1)
        monthly_revenue_input = gr.Dropdown(
            label="Monthly Revenue Range*",
            choices=["< 5 Lakh", "< 10 Lakh", "10-50 Lakh", "50 Lakh - 1 Crore", "> 1 Crore"],
            value="< 10 Lakh"
        )

        with gr.Row():
            back3_btn = gr.Button("← Back")
            next3_btn = gr.Button("Submit Business Profile", variant="primary") # Renamed button
        error3_bp = gr.Markdown() # Renamed error output for clarity
        output_html_step3 = gr.Markdown() # New markdown output for success message

    # Step 4: Data Upload
    with gr.Column(visible=False) as step4_data_upload_col:
        gr.Markdown("## Step 4 Upload Business Data")
        consent_check = gr.Checkbox(label="I consent to data analysis*", value=False)
        file_upload = gr.File(label="Upload Excel File*")
        file_upload_status = gr.Markdown(visible=False)

        with gr.Row():
            back4_btn = gr.Button("← Back")
            analyze_btn = gr.Button("🚀 Analyze Data", variant="primary")
            # Removed view_storyboard_btn = gr.Button("🎬 View Storyboard", visible=True) # New button
        error4_du = gr.Markdown() # Renamed error output for clarity

        insights_output = gr.Markdown()
        pdf_output = gr.File(label="Download PDF Report")
        view_dashboard_btn = gr.Button("📊 View Dashboard", visible=False, variant="primary")
        # Removed storyboard_output = gr.Markdown(visible=False) # New markdown output

    # Step 5: Dashboard
    with gr.Column(visible=False) as step5_dashboard_col:
        gr.Markdown("## 📊 Business Dashboard")

        with gr.Row():
            kpi1 = gr.Markdown("### 💰 Sales\n—")
            kpi2 = gr.Markdown("### 📈 Profit\n—")
            kpi3 = gr.Markdown("### 🧠 Health\n—")
            kpi4 = gr.Markdown("### 🚀 Growth\n—")

        with gr.Row():
            chart1 = gr.Plot()
            chart2 = gr.Plot()

        with gr.Row():
            chart3 = gr.Plot()
            chart4 = gr.Plot()

        back5_btn = gr.Button("⬅ Back to Data Upload")

    # ==================== HELPER FUNCTIONS ====================

    def update_visibility(step):
        return [
            gr.update(visible=(step == 0)),  # step0_landing_col
            gr.update(visible=(step == 1)),  # step1_user_info_col
            gr.update(visible=(step == 2)),  # step2_verification_col
            gr.update(visible=(step == 3)),  # step3_business_profile_col
            gr.update(visible=(step == 4)),  # step4_data_upload_col
            gr.update(visible=(step == 5))    # step5_dashboard_col
        ]

    def handle_login_and_visibility(mobile):
        profile = get_user_profile(mobile)
        if profile:
            new_step = 4
            return (f"✅ Welcome back, {profile['full_name']}!", profile, new_step, *update_visibility(new_step))
        return ("❌ No account found. Please register.", {}, 0, *update_visibility(0))

    def handle_register_and_visibility():
        new_step = 1
        return "", new_step, *update_visibility(new_step)

    def handle_cancel1_and_visibility():
        new_step = 0
        return new_step, *update_visibility(new_step)

    def validate_step1_and_visibility(name, mobile, role):
        if not name or not mobile or not role:
            return ("⚠️ Please fill all required fields", 1, *update_visibility(1))
        if not name.replace(' ', '').isalpha():
            return ("⚠️ Full Name should only contain letters and spaces.", 1, *update_visibility(1))
        if not mobile.isdigit():
            return ("⚠️ Mobile Number should only contain digits.", 1, *update_visibility(1))
        new_step = 2
        return "", new_step, *update_visibility(new_step)

    def handle_back_to_step1_and_visibility():
        new_step = 1
        return new_step, *update_visibility(new_step)

    def step2_verification_logic(msme_checked, consent_checked, cert_file, progress=gr.Progress()):
        if msme_checked and not consent_checked:
            return gr.update(value="⚠️ Please provide consent to share MSME details.", visible=True), gr.update(visible=False)
        if msme_checked and cert_file is None:
            # Corrected logic for mandatory file upload
            return gr.update(value="⚠️ Please upload your MSME certificate as it's mandatory if you are a registered MSME.", visible=True), gr.update(visible=False)

        progress(0, desc="Certificate submitted! 🏛️ Pending verification...")

        for i in progress.tqdm(range(5), desc="Verifying..."):
            time.sleep(1)

        verification_status = "VERIFIED" if msme_checked else "NOT_MSME"
        final_message = ""
        if verification_status == "VERIFIED":
            final_message = "<span style='color:green; font-weight:bold;'>✅ Your MSME verification is approved!</span>"
            return gr.update(value=final_message, visible=True), gr.update(visible=True)
        else:
            final_message = f"<span style='color:orange;'>ℹ️ Verification status: {verification_status}.</span> Still, you can proceed with data analysis."
            return gr.update(value=final_message, visible=True), gr.update(visible=True)

    def handle_continue_to_step3_and_visibility():
        new_step = 3
        return new_step, *update_visibility(new_step)

    def update_cities(state):
        if state == "Choose State":
            return gr.update(choices=["Choose City"], value="Choose City")
        if state in states_cities:
            return gr.update(choices=["Choose City"] + states_cities[state])
        return gr.update(choices=["Choose City"])

    def handle_back_to_step2_and_visibility():
        new_step = 2
        return new_step, *update_visibility(new_step)

    def validate_step3_and_visibility(current_data, company, biz_type, state, city, years_op, monthly_rev):
        if not company or biz_type == "Choose Business Type" or state == "Choose State" or city == "Choose City" or not years_op or not monthly_rev:
            return ("⚠️ Please fill all required fields", current_data, 3, *update_visibility(3), "") # Empty string for output_html_step3

        current_data.update({
            'company_name': company,
            'business_type': biz_type,
            'state': state,
            'city': city,
            'years_operation': years_op,
            'monthly_revenue_range': monthly_rev
        })

        success_message = f"""
<div style='text-align:center; font-size:20px; color:green; line-height:1.5;'>
    ✅ Business Details Submitted Successfully!<br><br>
    <b>Company Name:</b> {company}<br>
    <b>Business Type:</b> {biz_type}<br>
    <b>State:</b> {state}<br>
    <b>City:</b> {city}<br>
    <b>Years of Operation:</b> {years_op}<br>
    <b>Monthly Revenue Range:</b> {monthly_rev}<br>
</div>
"""
        new_step = 4
        return "", current_data, new_step, *update_visibility(new_step), success_message


    def handle_back_to_step3_and_visibility():
        new_step = 3
        return new_step, *update_visibility(new_step)

    def handle_view_dashboard_and_visibility():
        new_step = 5
        return new_step, *update_visibility(new_step)

    def handle_back_to_step4_and_visibility():
        new_step = 4
        return new_step, *update_visibility(new_step)

    def handle_analyze(user_data, consent, file):
        if not consent:
            return (
                "⚠️ Please provide consent",
                None,
                gr.update(visible=False),
                "N/A", "N/A", "N/A", "N/A",
                None, None, None, None,
                None, # Error message
                {} # Empty dict for analysis_summary_state
            )

        if file is None:
            return (
                "⚠️ Please upload a file",
                None,
                gr.update(visible=False),
                "N/A", "N/A", "N/A", "N/A",
                None, None, None, None,
                None, # Error message
                {} # Empty dict for analysis_summary_state
            )

        try:
            df = pd.read_excel(file.name)
            df_processed = calculate_scores(df) # Calculate scores once

            # Get raw numbers for storyboard and KPIs
            total_sales_raw = df_processed['Monthly_Sales_INR'].sum()
            avg_margin_raw = df_processed['Avg_Margin_Percent'].mean() if 'Avg_Margin_Percent' in df_processed.columns else 0
            total_profit_raw = total_sales_raw * (avg_margin_raw / 100)
            health_score_raw = df_processed['MSME_Health_Score'].mean()

            # Generate insights
            insights, error_insights, forecast_results = generate_insights(user_data, df_processed)
            if error_insights:
                return (error_insights, None, gr.update(visible=False),
                       "N/A", "N/A", "N/A", "N/A",
                       None, None, None, None, None, {}) # Added empty dict for analysis_summary_state

            # Generate dashboard data (formatted KPIs and charts)
            kpi_sales_str, kpi_profit_str, kpi_health_str, kpi_growth_str, fig1, fig2, fig3, fig4, dashboard_error_msg = generate_dashboard_data(user_data, df_processed)

            # Store summary for storyboard
            summary_for_storyboard = {
                'total_sales': total_sales_raw,
                'total_profit': total_profit_raw,
                'health_score': health_score_raw,
                'forecast_6m_sales': forecast_results['6_month']['forecast'],
                'avg_margin': avg_margin_raw
            }

            # Generate PDF, passing dashboard figures and error
            pdf_path, pdf_error = generate_pdf_report(user_data, df_processed,
                                                      dashboard_figs=[fig1, fig2, fig3, fig4],
                                                      dashboard_error_message=dashboard_error_msg)

            # Handle PDF generation errors
            if pdf_error:
                insights_with_pdf_error = f"{insights}\n\n**PDF Report Generation Error:** {pdf_error}"
                return (
                    insights_with_pdf_error,
                    None, # No PDF to download
                    gr.update(visible=False), # Hide view dashboard button if PDF failed badly
                    kpi_sales_str, kpi_profit_str, kpi_health_str, kpi_growth_str,
                    fig1, fig2, fig3, fig4,
                    dashboard_error_msg, # Pass original dashboard error if any
                    summary_for_storyboard # Pass summary
                )

            # If PDF was successful but dashboard had errors, update insights
            if dashboard_error_msg:
                insights_with_dashboard_error = f"{insights}\n\n**Dashboard Generation Error:** {dashboard_error_msg}"
                return (
                    insights_with_dashboard_error,
                    pdf_path, # PDF was successful
                    gr.update(visible=True), # View dashboard button still visible if PDF worked
                    "N/A", "N/A", "N/A", "N/A", # Clear dashboard KPIs and charts on UI if dashboard had errors
                    None, None, None, None,
                    dashboard_error_msg,
                    summary_for_storyboard # Pass summary
                )

            return (
                insights,
                pdf_path,
                gr.update(visible=True),
                kpi1, kpi2,
                kpi3, kpi4,
                fig1, fig2,
                fig3, fig4,
                dashboard_error_msg,
                summary_for_storyboard # New output
            )

        except Exception as e:
            return (
                f"❌ An unexpected error occurred during analysis: {str(e)}",
                None,
                gr.update(visible=False),
                "N/A", "N/A", "N/A", "N/A",
                None, None, None, None,
                None, # Error message
                {} # Empty dict for analysis_summary_state
            )

    def show_storyboard(summary_data):
        current_total_profit = summary_data.get('total_profit', 0)
        health_score = summary_data.get('health_score', 0)
        forecast_6m_sales = summary_data.get('forecast_6m_sales', 0)
        avg_margin = summary_data.get('avg_margin', 0)

        # Calculate projected profit for ACT 4
        projected_6m_profit = forecast_6m_sales * (avg_margin / 100)

        # Handle division by zero for percentage increase
        percentage_increase = 0
        if current_total_profit != 0:
            percentage_increase = ((projected_6m_profit - current_total_profit) / current_total_profit) * 100
        elif projected_6m_profit != 0: # If current profit is 0 and projected is positive
             percentage_increase = float('inf') # Indicate massive increase from zero

        # Adjust text for profit/loss if current_total_profit is negative or zero
        act1_profit_status = "Estimated Loss" if current_total_profit < 0 else "Current Profit"
        act1_profit_value = abs(current_total_profit) # Display absolute value

        act4_before_text = f"Before (Current Profit): ₹{current_total_profit:,.0f}"
        act4_after_text = f"After (Projected 6M Profit): ₹{projected_6m_profit:,.0f}"
        act4_increase_text = f"Increase: {percentage_increase:.1f}%"
        if percentage_increase == float('inf'):
            act4_increase_text = "Massive Increase (from zero/negative to positive)"
        elif percentage_increase < 0:
            act4_increase_text = f"Decrease: {abs(percentage_increase):.1f}%"


        storyboard_text = f"""
## 💡 AI Revelations - The Storyboard

### 😰 ACT 1: The Challenge
- No visibility into future
- Hidden profit drainers
- {act1_profit_status}: ₹{act1_profit_value:,.0f}

### 🔍 ACT 2: The Discovery
- Health Score: {health_score:.1f}%
- Transactions: 450 (Example: Actual transaction count would be dynamic if provided in data)

### 💡 ACT 3: AI Revelations
- Champions: Rice, Oil, Atta (Example: Top SKUs would be dynamic if extracted)
- Low margin: Snacks, Frozen (Example: Low margin SKUs would be dynamic if extracted)
- 6M Sales Forecast: ₹{forecast_6m_sales:,.0f}

### 🚀 ACT 4: Transformation
{act4_before_text} → {act4_after_text}
{act4_increase_text}
"""
        return gr.update(value=storyboard_text, visible=True)


    # ==================== EVENT HANDLERS ====================

    # === Step 0: Landing Page Events ===
    login_btn.click(
        handle_login_and_visibility,
        inputs=[login_mobile],
        outputs=[
            login_status,
            user_data_state,
            step_state,
            step0_landing_col,
            step1_user_info_col,
            step2_verification_col,
            step3_business_profile_col,
            step4_data_upload_col,
            step5_dashboard_col
        ]
    )

    register_btn.click(
        handle_register_and_visibility,
        inputs=[],
        outputs=[
            login_status,
            step_state,
            step0_landing_col,
            step1_user_info_col,
            step2_verification_col,
            step3_business_profile_col,
            step4_data_upload_col,
            step5_dashboard_col
        ]
    )

    # === Step 1: User Information Events ===
    cancel1_btn.click(
        handle_cancel1_and_visibility,
        inputs=[],
        outputs=[
            step_state,
            step0_landing_col,
            step1_user_info_col,
            step2_verification_col,
            step3_business_profile_col,
            step4_data_upload_col,
            step5_dashboard_col
        ]
    )

    next1_btn.click(
        lambda n, m, e, r: {**user_data_state.value, 'full_name': n, 'mobile_number': m, 'email': e, 'role': r},
        inputs=[name_input, mobile_input, email_input, role_input],
        outputs=[user_data_state] # Update user_data_state
    ).then(
        validate_step1_and_visibility,
        inputs=[name_input, mobile_input, role_input],
        outputs=[
            error1,
            step_state,
            step0_landing_col,
            step1_user_info_col,
            step2_verification_col,
            step3_business_profile_col,
            step4_data_upload_col,
            step5_dashboard_col
        ]
    ).then(
        lambda mobile: mobile, # Pass mobile_input to next function
        inputs=[mobile_input],
        outputs=[mobile_state] # Store mobile number in mobile_state
    ).then(
        lambda name: name, # Pass name_input to next function
        inputs=[name_input],
        outputs=[user_name_state] # Store name in user_name_state
    )

    # === Step 2: MSME Verification Events ===
    back2_btn.click(
        handle_back_to_step1_and_visibility,
        inputs=[],
        outputs=[
            step_state,
            step0_landing_col,
            step1_user_info_col,
            step2_verification_col,
            step3_business_profile_col,
            step4_data_upload_col,
            step5_dashboard_col
        ]
    )

    submit_btn_step2.click(
        step2_verification_logic,
        inputs=[msme_checkbox, consent_checkbox, certificate_file],
        outputs=[status_output_step2, next_step_btn_step2]
    )

    next_step_btn_step2.click(
        handle_continue_to_step3_and_visibility,
        inputs=[],
        outputs=[
            step_state,
            step0_landing_col,
            step1_user_info_col,
            step2_verification_col,
            step3_business_profile_col,
            step4_data_upload_col,
            step5_dashboard_col
        ]
    )

    # === Step 3: Business Profile Events ===
    state_input.change(
        update_cities,
        inputs=[state_input],
        outputs=[city_input]
    )

    back3_btn.click(
        handle_back_to_step2_and_visibility,
        inputs=[],
        outputs=[
            step_state,
            step0_landing_col,
            step1_user_info_col,
            step2_verification_col,
            step3_business_profile_col,
            step4_data_upload_col,
            step5_dashboard_col
        ]
    )

    next3_btn.click(
        validate_step3_and_visibility,
        inputs=[user_data_state, company_input, business_type_input,
               state_input, city_input, years_input, monthly_revenue_input],
        outputs=[
            error3_bp,
            user_data_state,
            step_state,
            step0_landing_col,
            step1_user_info_col,
            step2_verification_col,
            step3_business_profile_col,
            step4_data_upload_col,
            step5_dashboard_col,
            output_html_step3 # New output for success message
        ]
    ).then(
        lambda d: save_user_profile(d), # Save profile to DB
        inputs=[user_data_state],
        outputs=[user_id_state] # Update user_id_state with the saved ID
    ).then(
        lambda company, business_type, state, city: (company, business_type, state, city),
        inputs=[company_input, business_type_input, state_input, city_input],
        outputs=[company_state, business_type_state, state_state, city_state] # Update state variables
    )

    # === Step 4: Data Upload Events ===
    back4_btn.click(
        handle_back_to_step3_and_visibility,
        inputs=[],
        outputs=[
            step_state,
            step0_landing_col,
            step1_user_info_col,
            step2_verification_col,
            step3_business_profile_col,
            step4_data_upload_col,
            step5_dashboard_col
        ]
    )

    # The function `handle_file_upload_change` is not defined in the provided code.
    # To avoid a NameError, this line will be commented out. If you intend to use
    # this functionality, please define `handle_file_upload_change`.
    # file_upload.change(
    #     handle_file_upload_change,
    #     inputs=[file_upload, user_name_state],
    #     outputs=[file_upload_status]
    # )

    analyze_btn.click(
        handle_analyze,
        inputs=[user_data_state, consent_check, file_upload],
        outputs=[
            insights_output, pdf_output, view_dashboard_btn,
            kpi1, kpi2, kpi3, kpi4,
            chart1, chart2, chart3, chart4,
            analysis_summary_state # New output: analysis_summary_state
        ]
    )

    # Removed view_storyboard_btn.click handler

    view_dashboard_btn.click(
        handle_view_dashboard_and_visibility,
        inputs=[],
        outputs=[
            step_state,
            step0_landing_col,
            step1_user_info_col,
            step2_verification_col,
            step3_business_profile_col,
            step4_data_upload_col,
            step5_dashboard_col
        ]
    )

    # === Step 5: Dashboard Events ===
    back5_btn.click(
        handle_back_to_step4_and_visibility,
        inputs=[],
        outputs=[
            step_state,
            step0_landing_col,
            step1_user_info_col,
            step2_verification_col,
            step3_business_profile_col,
            step4_data_upload_col,
            step5_dashboard_col
        ]
    )

if __name__ == "__main__":
    print("=" * 60)
    print("🚀 MSME Intelligent Agent - Enhanced with Real ML")
    print("=" * 60)
    print("✅ Database: SQLite (msme_data.db)")
    print("✅ ML Models: Prophet, K-Means, Random Forest")
    print("✅ Features: Real forecasting, segmentation, analytics")
    print("=" * 60)
    demo.launch(share=False)

🚀 MSME Intelligent Agent - Enhanced with Real ML
✅ Database: SQLite (msme_data.db)
✅ ML Models: Prophet, K-Means, Random Forest
✅ Features: Real forecasting, segmentation, analytics
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>